In [1]:
import random
import string


In [8]:
lines = read_poem('/content/robert_frost.txt')
print(lines[:5])

['Two roads diverged in a yellow wood,', 'And sorry I could not travel both', 'And be one traveler, long I stood', 'And looked down one as far as I could', 'To where it bent in the undergrowth;']


In [9]:
def read_poem(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]


In [10]:
def clean_line(line):
    line = line.translate(str.maketrans('', '', string.punctuation))
    return line.lower().split()


In [11]:
def train_markov_model(lines):
    initial_counts = {}
    first_order_counts = {}
    second_order_counts = {}

    for line in lines:
        words = clean_line(line)
        if not words:
            continue

        first_word = words[0]
        initial_counts[first_word] = initial_counts.get(first_word, 0) + 1

        for i in range(len(words) - 1):
            w1, w2 = words[i], words[i + 1]
            if w1 not in first_order_counts:
                first_order_counts[w1] = {}
            first_order_counts[w1][w2] = first_order_counts[w1].get(w2, 0) + 1

        for i in range(len(words) - 2):
            w1, w2, w3 = words[i], words[i + 1], words[i + 2]
            pair = (w1, w2)
            if pair not in second_order_counts:
                second_order_counts[pair] = {}
            second_order_counts[pair][w3] = second_order_counts[pair].get(w3, 0) + 1

    return initial_counts, first_order_counts, second_order_counts


In [12]:
def normalize_counts(counts):
    probs = {}
    for key, subdict in counts.items():
        total = sum(subdict.values())
        probs[key] = {k: v / total for k, v in subdict.items()}
    return probs


In [13]:
def choose_word(prob_dict):
    r = random.random()
    cumulative = 0.0
    for word, prob in prob_dict.items():
        cumulative += prob
        if r <= cumulative:
            return word
    return random.choice(list(prob_dict.keys()))


In [14]:
def generate_poem(initial_probs, first_order, second_order, lines=4, max_words=10):
    poem = []
    for _ in range(lines):
        line = []
        first_word = choose_word(initial_probs)
        line.append(first_word)

        while len(line) < max_words:
            if len(line) == 1:
                prev = line[-1]
                if prev in first_order:
                    next_word = choose_word(first_order[prev])
                else:
                    break
            else:
                pair = (line[-2], line[-1])
                if pair in second_order:
                    next_word = choose_word(second_order[pair])
                else:
                    break
            line.append(next_word)

        poem.append(' '.join(line))
    return '\n'.join(poem)


In [17]:
if __name__ == "__main__":
    lines = read_poem('robert_frost.txt')
    if not lines:
        print("No lines found!")
    else:
        init, first, second = train_markov_model(lines)

        total_init = sum(init.values())
        init_probs = {k: v / total_init for k, v in init.items()}
        first_probs = normalize_counts(first)
        second_probs = normalize_counts(second)

        poem = generate_poem(init_probs, first_probs, second_probs)
        print("Generated Poem\n")
        print(poem)


Generated Poem

i stole the goblet from the childrens playhouse
back its worse than that she cant
the chimney started above the stove
or the hall door for the clock
